In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, GridSearchCV
from sklearn.linear_model import ElasticNet, ElasticNetCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from math import sqrt
import mysql.connector

In [ ]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database="avito_db"
)

cursor = conn.cursor()

In [ ]:
import warnings
warnings.filterwarnings('ignore')


In [ ]:
query = "SELECT * FROM FinalDataSet_bureaux"
df = pd.read_sql(query, conn)
df.head()

In [ ]:
# Dataframe without target variable:
X = df.drop(columns = ["Prix", "Lien", "Age", "Condition", "Quartier", "Ville"])
# Target variable:
y = df['Prix']
# Splitting the data:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=5)

<h2 style="color:black"> ElasticNet </h2>

In [ ]:
scaler = StandardScaler()

# Fit the scaler to the training data and transform it
X_train_scaled = scaler.fit_transform(X_train)

# Define the grid of values for α and l1_ratio
alphas = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
l1_ratios = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

# Initialize the ElasticNetCV model
model_ElasticNetCV = ElasticNetCV(alphas=alphas, l1_ratio=l1_ratios, cv=10)

# Fit the model to the scaled training data
model_ElasticNetCV.fit(X_train_scaled, y_train)

# The optimal values of α and l1_ratio are stored in `alpha_` and `l1_ratio_` attributes of the model
optimal_alpha = model_ElasticNetCV.alpha_
optimal_l1_ratio = model_ElasticNetCV.l1_ratio_

In [ ]:
print(optimal_alpha)
print(optimal_l1_ratio)

In [ ]:
model_ElasticNet = ElasticNet(alpha=0.5, l1_ratio=0.9)

# Perform 10-fold cross-validation on the scaled training data
scores = cross_val_score(model_ElasticNet, X_train_scaled, y_train, cv=10)

# Calculate the average R² score
avg_r2_ElasticNet = scores.mean()

# Generate cross-validated predictions on the scaled training data
cv_predictions_ElasticNet = cross_val_predict(model_ElasticNet, X_train_scaled, y_train, cv=10)

In [ ]:
cv_predictions_ElasticNet

In [ ]:
#Validation:
cv_rmse_ElasticNet = np.sqrt(mean_squared_error(y_train, cv_predictions_ElasticNet))

# Training:
model_ElasticNet.fit(X_train, y_train)
y_train_pred = model_ElasticNet.predict(X_train)
r2_training_ElasticNet = r2_score(y_train, y_train_pred)
train_rmse_ElasticNet = sqrt(mean_squared_error(y_train, y_train_pred))

#____test_____
y_test_pred_ElasticNet = model_ElasticNet.predict(X_test)
r2_test_ElasticNet = r2_score(y_test,y_test_pred_ElasticNet)
test_rmse_ElasticNet = sqrt(mean_squared_error(y_test, y_test_pred_ElasticNet))

In [ ]:
print(f'Validation R² for ElasticNet: {avg_r2_ElasticNet:.2f}')
print(f'Validation RMSE for ElasticNet: {cv_rmse_ElasticNet:.2f}')
print("________________________")
print(f'Training R² for ElasticNet: {r2_training_ElasticNet:.2f}')
print(f'Training RMSE for ElasticNet: {train_rmse_ElasticNet:.2f}')
print("________________________")
print(f'Test R² for ElasticNet: {r2_test_ElasticNet:.2f}')
print(f'Test RMSE for ElasticNet: {test_rmse_ElasticNet:.2f}')

In [ ]:
# Convert to numpy arrays for easy sorting
y_test_np = np.array(y_test)
y_test_pred_np = np.array(y_test_pred_ElasticNet)

# Get the sorted order of y_test
sort_order = np.argsort(y_test_np)

# Apply the sort order to both y_test and y_test_pred
y_test_sorted = y_test_np[sort_order]
y_test_pred_sorted = y_test_pred_np[sort_order]

plt.figure(figsize=(6,4))
plt.scatter(y_test_sorted, y_test_pred_sorted, color='#0080ff', s=4) 
plt.plot(y_test_sorted, y_test_pred_sorted, color='orange')
plt.title('ElasticNet Model Predictions vs Test Data')
plt.ylabel('Actual Price')
plt.xlabel('Price Prediction')
plt.show()

<h2 style="color:black"> Decision Tree Regression </h2>

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid
param_grid = {
    'max_depth': [ 10, 20, 30, 40],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Create a GridSearchCV object
grid = GridSearchCV(DecisionTreeRegressor(random_state=0), param_grid, refit=True, verbose=2, cv=10)

# Fit the model to the data
grid.fit(X_train, y_train)

# Print the best parameters
print(grid.best_params_)

In [ ]:
modelDTR = DecisionTreeRegressor(max_depth = 20, min_samples_leaf = 2, min_samples_split = 10, random_state=44)

In [ ]:
# Calculate the Validation R² and rmse score
scores = cross_val_score(modelDTR, X_train, y_train, cv=10)
avg_r2_tree = scores.mean()
cv_predictions_tree = cross_val_predict(modelDTR, X_train, y_train, cv=10)
cv_rmse_tree = np.sqrt(mean_squared_error(y_train, cv_predictions_tree))
# Calculate the training R² and rmse
modelDTR.fit(X_train, y_train)
y_train_pred_tree = modelDTR.predict(X_train)
r2_training_tree = r2_score(y_train, y_train_pred_tree)
y_train_pred_tree  = modelDTR.predict(X_train)
train_rmse_tree = sqrt(mean_squared_error(y_train, y_train_pred_tree))
# Calculate the test R² and rmse:
test_predictions = modelDTR.predict(X_test)
test_r2_tree = r2_score(y_test, test_predictions)
test_rmse3_tree = sqrt(mean_squared_error(y_test, test_predictions))

In [ ]:
print(f'Validation R² for Decision Tree Regression: {avg_r2_tree:.2f}')
print(f'Validation RMSE for Decision Tree Regression : {cv_rmse_tree:.2f}')
print("___________________________________")
print(f'Training R² for Decision Tree Regression: {r2_training_tree:.2f}')
print(f'Training RMSE for Decision Tree Regression: {train_rmse_tree:.2f}')
print("___________________________________")
print(f'Test R² for Decision Tree Regression: {test_r2_tree:.2f}')
print(f'Test RMSE for Decision Tree Regression: {test_rmse3_tree:.2f}')

In [ ]:
y_test_pred_np = np.array(test_predictions)

# Apply the sort order to both y_test and y_test_pred
y_test_sorted1 = y_test_np[sort_order]
y_test_pred_sorted1 = y_test_pred_np[sort_order]

plt.figure(figsize=(6,4))
plt.scatter(y_test_sorted1, y_test_pred_sorted1, color='#0080ff', s=4) 
plt.plot(y_test_sorted1, y_test_pred_sorted1, color='orange')
plt.title('Decision Tree Regression Model Predictions vs Test Data')
plt.ylabel('Actual Price')
plt.xlabel('Price Prediction')
plt.show()

<h2 style="color:black"> Random Forest Regression </h2>

In [ ]:
# # Define the parameter grid
param_grid = {
     'n_estimators': [100, 200 ],
     'max_depth': [None, 10, 20, 40],
     'min_samples_split': [2, 5],
     'min_samples_leaf': [1, 4]
}

# Create a GridSearchCV object
grid = GridSearchCV(RandomForestRegressor(random_state=0), param_grid, refit=True, verbose=2, cv=10)

# Fit the model to the data
grid.fit(X_train, y_train)

# Print the best parameters
print(grid.best_params_)

# Fit the model with the best parameters to the training data
model = RandomForestRegressor(random_state=0, **grid.best_params_)

In [ ]:
modelRFR = RandomForestRegressor(max_depth=None, min_samples_split=5, min_samples_leaf=1, n_estimators=100,random_state=0)

In [ ]:
# Calculate the Validation R² and rmse score
scores = cross_val_score(modelRFR, X_train, y_train, cv=10)
avg_r2_rfr = scores.mean()
cv_predictions = cross_val_predict(modelRFR, X_train, y_train, cv=10)
cv_rmse_rfr = np.sqrt(mean_squared_error(y_train, cv_predictions))
# Calculate the training R² and rmse
modelRFR.fit(X_train, y_train)
y_train_pred_rfr = modelRFR.predict(X_train)
r2_training_rfr = r2_score(y_train, y_train_pred_rfr)
y_train_pred_rfr = modelRFR.predict(X_train)
train_rmse_rfr = sqrt(mean_squared_error(y_train, y_train_pred_rfr))

# Calculate the test R² and rmse:
test_predictions2 = modelRFR.predict(X_test)
test_r2_rfr = r2_score(y_test, test_predictions2)
test_rmse_rfr = sqrt(mean_squared_error(y_test, test_predictions2))

In [ ]:
print(f'Validation R² for Random Forest Regression: {avg_r2_rfr:.2f}')
print(f'Validation RMSE for Random Forest Regression : {cv_rmse_rfr:.2f}')
print("___________________________________")
print(f'Training R² for Random Forest Regression: {r2_training_rfr:.2f}')
print(f'Training RMSE for Random Forest Regression: {train_rmse_rfr:.2f}')
print("___________________________________")
print(f'Test R² for Random Forest Regression: {test_r2_rfr:.2f}')
print(f'Test RMSE for Random Forest Regression: {test_rmse_rfr:.2f}')

In [ ]:
y_test_pred_np2  = np.array(test_predictions2)
# Apply the sort order to both y_test and y_test_pred
y_test_sorted2 = y_test_np[sort_order]
y_test_pred_sorted2 = y_test_pred_np2[sort_order]

plt.figure(figsize=(11,4))
plt.scatter(y_test_sorted2, y_test_pred_sorted2, color='#0080ff', s=4) 
plt.plot(y_test_sorted2, y_test_pred_sorted2, color='orange')
plt.title('Random Forest Model Predictions vs Test Data')
plt.ylabel('Actual Price')
plt.xlabel('Price Prediction')
plt.savefig("model.png")
plt.show()

<h2 style="color:black"> Gradient Boosting Regression </h2>

In [ ]:
# Define the parameter grid
param_grid = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.1, 0.05, 0.02, 0.01],
    'max_depth': [4, 6],
    'min_samples_leaf': [3, 5, 9, 17],
    'max_features': [1.0, 0.3, 0.1]
}

# Create a GridSearchCV object
grid = GridSearchCV(GradientBoostingRegressor(random_state=0), param_grid, cv=10, n_jobs=-1)

# Fit the model to the data
grid.fit(X_train, y_train)

# Print the best parameters
print(grid.best_params_)

In [ ]:
modelGB = GradientBoostingRegressor(learning_rate=  0.05, max_depth = 4, max_features= 0.3, min_samples_leaf = 3, n_estimators = 200,random_state=0)
modelGB.fit(X_train, y_train)

In [ ]:
# Calculate the Validation R² and rmse score
scores = cross_val_score(modelGB, X_train, y_train, cv=10)
avg_r2_gb = scores.mean()
cv_predictions = cross_val_predict(modelGB, X_train, y_train, cv=10)
cv_rmse_gb = np.sqrt(mean_squared_error(y_train, cv_predictions))
# Calculate the training R² and rmse
y_train_pred_gb= modelGB.predict(X_train)
r2_training_gb = r2_score(y_train, y_train_pred_gb)
y_train_pred_gb = modelGB.predict(X_train)
train_rmse_gb = sqrt(mean_squared_error(y_train, y_train_pred_gb))
# Calculate the test R² and rmse:
test_predictions = modelGB.predict(X_test)
test_r2_gb = r2_score(y_test, test_predictions)
test_rmse_gb = sqrt(mean_squared_error(y_test, test_predictions))

In [ ]:
print(f'Validation R² for Gradient Boosting: {avg_r2_gb:.2f}')
print(f'Validation RMSE for Gradient Boosting : {cv_rmse_gb:.2f}')
print("___________________________________")
print(f'Training R² for Gradient Boosting: {r2_training_gb:.2f}')
print(f'Training RMSE for Gradient Boosting: {train_rmse_gb:.2f}')
print("___________________________________")
print(f'Test R² for Decision Gradient Boosting: {test_r2_gb:.2f}')
print(f'Test RMSE for Decision Gradient Boosting: {test_rmse_gb:.2f}')

In [ ]:
y_test_pred_np3  = np.array(test_predictions)

# Get the sorted order of y_test
sort_order = np.argsort(y_test_np)

# Apply the sort order to both y_test and y_test_pred
y_test_sorted3 = y_test_np[sort_order]
y_test_pred_sorted3 = y_test_pred_np3[sort_order]

plt.figure(figsize=(6,4))
plt.scatter(y_test_sorted3, y_test_pred_sorted3, color='#0080ff', s=4) 
plt.plot(y_test_sorted3, y_test_pred_sorted3, color='orange')
plt.title('Gradient Boosting Model Predictions vs Test Data')
plt.ylabel('Actual Price')
plt.xlabel('Price Prediction')
plt.show()

<h2 style="color:black"> XGB Regression </h2>

In [ ]:
from xgboost import XGBRegressor

# Define the parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.7, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.9, 1.0]
}

# Create a GridSearchCV object
grid = GridSearchCV(XGBRegressor(random_state=0), param_grid, cv=10, n_jobs=-1)

# Fit the model to the data
grid.fit(X_train, y_train)

# Print the best parameters
print(grid.best_params_)

In [ ]:
from xgboost import XGBRegressor

modelXGB = XGBRegressor(colsample_bytree = 0.9, learning_rate = 0.05, max_depth = 3, n_estimators = 200, subsample = 0.7,random_state=0)
modelXGB.fit(X_train, y_train)

In [ ]:
# Calculate the Validation R² and rmse score
scores = cross_val_score(modelXGB, X_train, y_train, cv=10)
avg_r2_xgb = scores.mean()
cv_predictions = cross_val_predict(modelXGB, X_train, y_train, cv=10)
cv_rmse_xgb = np.sqrt(mean_squared_error(y_train, cv_predictions))
# Calculate the training R² and rmse
y_train_pred_xgb= modelXGB.predict(X_train)
r2_training_xgb = r2_score(y_train, y_train_pred_xgb)
y_train_pred_xgb = modelXGB.predict(X_train)
train_rmse_xgb = sqrt(mean_squared_error(y_train, y_train_pred_xgb))
# Calculate the test R² and rmse:
test_predictions = modelXGB.predict(X_test)
test_r2_xgb = r2_score(y_test, test_predictions)
test_rmse_xgb = sqrt(mean_squared_error(y_test, test_predictions))

In [ ]:
print(f'Validation R² for Gradient Boosting: {avg_r2_xgb:.2f}')
print(f'Validation RMSE for Gradient Boosting : {cv_rmse_xgb:.2f}')
print("___________________________________")
print(f'Training R² for Gradient Boosting: {r2_training_xgb:.2f}')
print(f'Training RMSE for Gradient Boosting: {train_rmse_xgb:.2f}')
print("___________________________________")
print(f'Test R² for Decision Gradient Boosting: {test_r2_xgb:.2f}')
print(f'Test RMSE for Decision Gradient Boosting: {test_rmse_xgb:.2f}')

<h2 style="color:black"> LightGBM </h2>

In [ ]:
from lightgbm import LGBMRegressor


# Define the parameter grid
param_grid = {
    'n_estimators': [100, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [4, 6, 8],
    'num_leaves': [31, 50, 100],
    'subsample': [0.8, 1.0]
}

# Create a GridSearchCV object
grid = GridSearchCV(LGBMRegressor(random_state=0), param_grid, cv=10, n_jobs=-1)

# Fit the model to the data
grid.fit(X_train, y_train)

# Print the best parameters
print(grid.best_params_)

In [ ]:
from lightgbm import LGBMRegressor
modelLGBM = LGBMRegressor(learning_rate = 0.05, max_depth = 6, n_estimators = 100, num_leaves = 31, subsample = 0.8, random_state=0)
modelLGBM.fit(X_train, y_train)

In [ ]:
# Calculate the Validation R² and rmse score
scores = cross_val_score(modelLGBM, X_train, y_train, cv=10)
avg_r2_lgbm = scores.mean()
cv_predictions = cross_val_predict(modelLGBM, X_train, y_train, cv=10)
cv_rmse_lgbm = np.sqrt(mean_squared_error(y_train, cv_predictions))
# Calculate the training R² and rmse
y_train_pred_lgbm= modelLGBM.predict(X_train)
r2_training_lgbm = r2_score(y_train, y_train_pred_lgbm)
y_train_pred_lgbm = modelLGBM.predict(X_train)
train_rmse_lgbm = sqrt(mean_squared_error(y_train, y_train_pred_lgbm))
# Calculate the test R² and rmse:
test_predictions = modelLGBM.predict(X_test)
test_r2_lgbm = r2_score(y_test, test_predictions)
test_rmse_lgbm = sqrt(mean_squared_error(y_test, test_predictions))

In [ ]:
print(f'Validation R² for Gradient Boosting: {avg_r2_lgbm:.2f}')
print(f'Validation RMSE for Gradient Boosting : {cv_rmse_lgbm:.2f}')
print("___________________________________")
print(f'Training R² for Gradient Boosting: {r2_training_lgbm:.2f}')
print(f'Training RMSE for Gradient Boosting: {train_rmse_lgbm:.2f}')
print("___________________________________")
print(f'Test R² for Decision Gradient Boosting: {test_r2_lgbm:.2f}')
print(f'Test RMSE for Decision Gradient Boosting: {test_rmse_lgbm:.2f}')

<h2 style="color:black"> Comparaison & Discussion </h2>

In [ ]:
# Assuming the R² values are as follows:
validation_rmse_values = [ cv_rmse_gb, cv_rmse_rfr, cv_rmse_tree, cv_rmse_ElasticNet, cv_rmse_xgb, cv_rmse_lgbm]
training_rmse_values = [ train_rmse_gb, train_rmse_rfr, train_rmse_tree, train_rmse_ElasticNet, train_rmse_xgb, train_rmse_lgbm]
test_rmse_values = [ test_rmse_gb, test_rmse_rfr, test_rmse3_tree, test_rmse_ElasticNet, test_rmse_xgb, test_rmse_lgbm]
models = [ 'Gradient Boosting', 'Random Forest', 'Decision Tree', "ElasticNet", 'XGBoost', 'LightGBM']

# Setting the positions and width for the bars
pos = list(range(len(models))) 
width = 0.25 

# Plotting the bars
fig, ax = plt.subplots(figsize=(11,4))

# Create a horizontal bar with validation_r2 data,
# in position pos,
plt.barh(pos, 
        validation_rmse_values, 
        width, 
        alpha=0.5, 
        color='green', 
        label='Validation RMSE') 

# Create a horizontal bar with training_r2 data,
# in position pos + some width buffer,
plt.barh([p + width for p in pos], 
        training_rmse_values, 
        width, 
        alpha=0.5, 
        color='blue', 
        label='Training RMSE') 

# Create a horizontal bar with test_r2 data,
# in position pos + 2*width buffer,
plt.barh([p + 2*width for p in pos], 
        test_rmse_values, 
        width, 
        alpha=0.5, 
        color='red', 
        label='Test RMSE') 

# Set the x axis label
ax.set_xlabel('RMSE')

# Set the chart's title
ax.set_title('Validation, Training, and Test RMSE for Different Models')

# Set the position of the y ticks
ax.set_yticks([p + 1.5 * width for p in pos])

# Set the labels for the y ticks
ax.set_yticklabels(models)

# Setting the x-axis and y-axis limits
plt.ylim(min(pos)-width, max(pos)+width*3)
plt.xlim([0, 900000])  # Set the limits of the x-axis here

# Adding the legend and showing the plot
plt.legend(loc='upper right')
plt.grid()

plt.show()

In [ ]:
# Assuming the R² values are as follows:
validation_r2_values = [ avg_r2_gb, avg_r2_rfr, avg_r2_tree, avg_r2_ElasticNet, avg_r2_xgb, avg_r2_lgbm]
training_r2_values = [ r2_training_gb, r2_training_rfr, r2_training_tree, r2_training_ElasticNet, r2_training_xgb, r2_training_lgbm]
test_r2_values = [ test_r2_gb, test_r2_rfr, test_r2_tree, r2_test_ElasticNet, test_r2_xgb, test_r2_lgbm]
models = [ 'Gradient Boosting', 'Random Forest', 'Decision Tree', "ElasticNet", 'XGBoost', 'LightGBM']

# Setting the positions and width for the bars
pos = list(range(len(models))) 
width = 0.25 

# Plotting the bars
fig, ax = plt.subplots(figsize=(11,4))

# Create a horizontal bar with validation_r2 data,
# in position pos,
plt.barh(pos, 
        validation_r2_values, 
        width, 
        alpha=0.5, 
        color='green', 
        label='Validation R²') 

# Create a horizontal bar with training_r2 data,
# in position pos + some width buffer,
plt.barh([p + width for p in pos], 
        training_r2_values, 
        width, 
        alpha=0.5, 
        color='blue', 
        label='Training R²') 

# Create a horizontal bar with test_r2 data,
# in position pos + 2*width buffer,
plt.barh([p + 2*width for p in pos], 
        test_r2_values, 
        width, 
        alpha=0.5, 
        color='red', 
        label='Test R²') 

# Set the x axis label
ax.set_xlabel('R²')

# Set the chart's title
ax.set_title('Validation, Training, and Test R² for Different Models')

# Set the position of the y ticks
ax.set_yticks([p + 1.5 * width for p in pos])

# Set the labels for the y ticks
ax.set_yticklabels(models)

# Setting the x-axis and y-axis limits
plt.ylim(min(pos)-width, max(pos)+width*3)
plt.xlim([0, 1])  # Set the limits of the x-axis here

# Adding the legend and showing the plot
plt.legend(loc='upper right')
plt.grid()
plt.show()

In [ ]:
print(models)
print("validation_r2", validation_r2_values)
print("training_r2", training_r2_values)
print("test_r2", test_r2_values)
print("validation_rms", validation_rmse_values)
print("training_rmse", training_rmse_values)
print("test_rmse", test_rmse_values)

<h2 style="color:black"> Stacking model </h2>

In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression

estimators = [
    ('en', ElasticNet(alpha=0.5, l1_ratio=0.9)) ,
    ('gb', GradientBoostingRegressor(learning_rate=  0.05, max_depth = 4, max_features= 0.3, min_samples_leaf = 3, n_estimators = 200,random_state=0)),
    ('xgb', XGBRegressor(colsample_bytree = 0.9, learning_rate = 0.05, max_depth = 3, n_estimators = 200, subsample = 0.7,random_state=0)),
]

stack_model = StackingRegressor(
    estimators=estimators,
    final_estimator=LinearRegression(),
    passthrough=False
)

stack_model.fit(X_train, y_train)

scores = cross_val_score(stack_model, X_train, y_train, cv=10)
avg_r2_stack = scores.mean()

cv_predictions = cross_val_predict(stack_model, X_train, y_train, cv=10)
cv_rmse_stack = np.sqrt(mean_squared_error(y_train, cv_predictions))

# Training
y_train_pred_stack = stack_model.predict(X_train)
r2_training_stack = r2_score(y_train, y_train_pred_stack)
train_rmse_stack = sqrt(mean_squared_error(y_train, y_train_pred_stack))

# Test
y_test_pred_stack = stack_model.predict(X_test)
r2_test_stack = r2_score(y_test, y_test_pred_stack)
test_rmse_stack = sqrt(mean_squared_error(y_test, y_test_pred_stack))

In [ ]:
print(f'Validation R² for Stackin Model: {avg_r2_stack:.2f}')
print(f'Validation RMSE for Stackin Model: {cv_rmse_stack:.2f}')
print("___________________________________")
print(f'Training R² for Stackin Model: {r2_training_stack:.2f}')
print(f'Training RMSE for Stackin Model: {train_rmse_stack:.2f}')
print("___________________________________")
print(f'Test R² for Stackin Model: {r2_test_stack:.2f}')
print(f'Test RMSE for Stackin Model: {test_rmse_stack:.2f}')


<h2 style="color:black"> Make a prediction using Stacking Model </h2>

In [ ]:
X_train.columns

In [ ]:
X_train.head(2)

In [ ]:
# 1️⃣ Mapping Quartier
quartier_mean_mapping = df.groupby("Quartier")["Prix"].mean().to_dict()
global_mean = df["Prix"].mean()

# ✅ Exemple user_input brut
user_input = {
    "Surface Totale": 79,
    "Salles De Bain": 1,
    "Etage": 0,
    "Terrasse": 1,
    "Parking": 1,
    "Sécurité": 1,
    "Chauffage": 0,
    "Climatisation": 1,
    "Ascenseur": 0,
    "Age_encoded": 0,
    "Condition_encoded": 0,
    "Quartier": "Haut-Founty",
    "Ville": "Agadir"   
}

# ✅ 1) Calculer Quartier_enc et l'ajouter
quartier_enc = quartier_mean_mapping.get(user_input["Quartier"], global_mean)
user_input["Quartier_enc"] = quartier_enc

# ✅ 2) Supprimer Quartier & Ville texte du dict :
del user_input["Quartier"]
del user_input["Ville"]

# ✅ 3) Détecter features
numerical_features = [
    "Surface Totale",
    "Salles De Bain",
    "Etage",
    "Terrasse",
    "Parking",
    "Sécurité",
    "Chauffage",
    "Climatisation",
    "Ascenseur",
    "Age_encoded",
    "Condition_encoded",
    "Quartier_enc"
]

ville_dummies = [col for col in df.columns if col.startswith("Ville_")]
features = numerical_features + ville_dummies

# ✅ 4) Construire Ville_* proprement
ville_dummies_input = dict.fromkeys(ville_dummies, 0)
ville_col = f"Ville_Marrakech"  # c'est ta valeur Ville
if ville_col in ville_dummies:
    ville_dummies_input[ville_col] = 1
else:
    ville_dummies_input["Ville_Autres"] = 1

# ✅ 5) Fusionner et construire X_new
X_new = pd.DataFrame([{**user_input, **ville_dummies_input}])

# ✅ 6) Réordonner sans erreur
X_new = X_new[features]

# ✅ 7) Prédire
y_pred = stack_model.predict(X_new)
print("✅ Prix prédit :", y_pred[0])


In [ ]:
import joblib

joblib.dump(stack_model, "model_bureaux.pkl")
